# Exploring FASERnu neutrino data

FASER is an experiment at CERN that studies **neutrinos** produced by the Large Hadron
Collider. Its detector, FASERnu, is a stack of ~700 photographic films. When a particle
passes through, it leaves a track in the emulsion.

When a neutrino hits a nucleus in the detector it produces a spray of charged particles
all starting from one point. That point is called a **vertex**, and the tracks coming out
of it are the **primary tracks**.

The problem: a neutrino is not the only thing that can make a vertex like this. A
**neutral hadron** (a neutron, a $\Lambda$, a $K^0$) is also electrically neutral, so it
also leaves no track coming *in*, and it also makes a spray of particles going *out*.
Under a microscope the two look very similar. Telling them apart using machine learning is the goal of this project.
This notebook showcases the basics of the Python libraries and the data.

- **Signal** = a real neutrino interaction
- **Background** = a neutral hadron interaction

Each row of `vertices.csv` is one vertex, already summarised from its tracks.

## 1. Setting up

If you have not installed the packages yet, see the `README.md` in the folder above.
These four lines import the tools we need. It is normal for the first import to take a
few seconds.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the plots a bit bigger and easier to read
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["font.size"] = 14

## 2. Loading the data

`pd.read_csv` reads a CSV file into a **DataFrame**, which is pandas' name for a table.

In [ ]:
df = pd.read_csv("data/vertices.csv")

print(f"The table has {len(df)} rows and {len(df.columns)} columns.")
df.head()

`.head()` shows the first five rows. Some useful first commands:

- `df.shape` - how many rows and columns
- `df.columns` - the column names
- `df.describe()` - minimum, maximum, mean and quartiles of every numeric column

Try them. `describe()` is wide, so scroll sideways.

In [ ]:
df.describe()

## 3. What kind of interaction is each row?

The `interaction` column says what actually happened. We know this because the data is
**simulated**, so the true answer was recorded when it was generated. In real data we
would not have this column - that is the whole point of building a classifier.

The five values mean:

| value | meaning |
|---|---|
| `numuCC` | a muon neutrino, charged-current (it turned into a muon) |
| `nueCC` | an electron neutrino, charged-current (it turned into an electron) |
| `nutauCC` | a tau neutrino, charged-current (rare!) |
| `NC` | any neutrino, neutral-current (the neutrino bounced off and kept going) |
| `background` | a neutral hadron - **not** a neutrino |

In [ ]:
df["interaction"].value_counts()

For classification we usually want a simple yes/no answer rather than five categories.
There is no ready-made column for this, on purpose - it is one line to make:

In [ ]:
# True for any kind of charged-current neutrino interaction, False for a neutral hadron and neutral current interactions
is_signal = (df["interaction"] != "background") & (df["interaction"] != "NC")

print(f"signal     : {is_signal.sum()}")
print(f"background : {(~is_signal).sum()}")

## 4. A warning about the `weight` column

**Never use `weight` as an input to the machine learning model.**

`weight` says how many real interactions each simulated one represents. It depends only
on *which simulation file* the row came from - and the signal and background came from
different files. So the weight gives away the answer:

In [ ]:
# Every weight value belongs to exactly one class - it is a perfect give-away
df.groupby("weight")["interaction"].unique()

So what is `weight` *for*? It converts "number of simulated events" into "number of
events we expect to actually see in the real detector". We will use it in section 7.

## 5. Your first histogram

A **histogram** counts how many rows fall into each range of values. Let's look at
`n_vtrk`, the number of primary tracks coming out of the vertex.

In [ ]:
fig, ax = plt.subplots(layout="constrained")

ax.hist(df["n_vtrk"], bins=np.arange(0.5, 40.5, 1))

ax.set_xlabel("Number of primary tracks")
ax.set_ylabel("Number of vertices")
ax.set_title("Track multiplicity of all vertices")
ax.grid(alpha=0.3)

Always label your axes. A plot without labels tells the reader nothing.

`bins=np.arange(0.5, 40.5, 1)` puts the bin edges at 0.5, 1.5, 2.5 ... so that each whole
number sits in the middle of its own bin. That matters for counting things.

## 6. Comparing signal and background

This is the important plot. If the two classes look different in some variable, a
classifier can use that variable. If they look identical, it cannot.

`density=True` rescales each histogram so its total area is 1. Without it the taller
histogram would just be the one with more rows, which tells us nothing. Feel free to try!

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")

# n_vtrk counts whole tracks, so its bins must be centred on whole numbers.
# delta_phi is continuous, so any reasonable number of bins works.
for ax, column, label, bins in [
    (axes[0], "n_vtrk", "Number of primary tracks", np.arange(0.5, 40.5, 1)),
    (axes[1], "delta_phi_p_mean", r"Mean $\Delta\phi$, momentum weighted [rad]", 30),
]:
    ax.hist(df.loc[is_signal, column].dropna(), bins=bins, density=True,
            histtype="step", linewidth=2, label="signal (neutrino)")
    ax.hist(df.loc[~is_signal, column].dropna(), bins=bins, density=True,
            histtype="step", linewidth=2, label="background (neutral hadron)")
    ax.set_xlabel(label)
    ax.set_ylabel("Fraction of vertices")
    ax.legend()
    ax.grid(alpha=0.3)

Two real physics effects are visible here:

**Left:** neutral hadrons produce *more* tracks than neutrinos. A hadron interacts via
the strong force, which is messier and sprays out more particles.

**Right:** `delta_phi_p_mean` measures how **back-to-back** the vertex is. Imagine looking
down the beam pipe: in a neutrino interaction the outgoing lepton flies one way and all
the other particles recoil the other way, so the angle between them is close to
$\pi$ (180 degrees). A neutral hadron has no lepton, so its tracks are spread more evenly
and the angle is smaller.

Try swapping in other columns - `p_mean`, `n_kinks`, `ip_pos_max`, `slope_max` - and see
which ones separate the two classes well.

## 7. Using the weights

The simulation contains roughly equal numbers of signal and background, but that is **not**
what the real experiment sees. Multiplying by `weight` tells us the real expectation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
bins = np.arange(0.5, 30.5, 1)

for ax, weights, title in [
    (axes[0], None, "As simulated"),
    (axes[1], df["weight"], "Scaled to the real experiment"),
]:
    ax.hist(df.loc[is_signal, "n_vtrk"], bins=bins, histtype="step", linewidth=2,
            weights=None if weights is None else weights[is_signal], label="signal")
    ax.hist(df.loc[~is_signal, "n_vtrk"], bins=bins, histtype="step", linewidth=2,
            weights=None if weights is None else weights[~is_signal], label="background")
    ax.set_xlabel("Number of primary tracks")
    ax.set_ylabel("Number of vertices" if weights is None else "Expected vertices")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

print(f"As simulated : {is_signal.sum():>8} signal, {(~is_signal).sum():>8} background")
print(f"Real expectation : {df.loc[is_signal, 'weight'].sum():>6.0f} signal, "
      f"{df.loc[~is_signal, 'weight'].sum():>6.0f} background")

This is the single most important plot in the notebook. As simulated the classes are
almost balanced. In the real experiment the background outnumbers the signal about
**8 to 1**. A classifier that looks good on the simulated numbers can still be useless in
practice, because every mistake on a background event costs eight times as much.

## 8. Two variables at once

A histogram shows one variable. To see how two variables behave together, use a
**2D histogram** plot: the plane is segmented into smaller rectangles and each is coloured by how many points
fall in it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained", sharey=True)

x_bins = np.arange(0.5, 30.5, 1)
y_bins = np.arange(0, 300, 10)

for ax, mask, title in [
    (axes[0], is_signal, "Signal (neutrino)"),
    (axes[1], ~is_signal, "Background (neutral hadron)"),
]:
    subset = df[mask]
    # bins="log" colours by the logarithm of the count. Without it the one busiest
    # hexagon is dark and everything else is nearly white.
    *_, image = ax.hist2d(subset["n_vtrk"], subset["p_mean"], bins=(x_bins, y_bins), cmap="Blues", cmin=1)
    ax.set_xlabel("Number of primary tracks")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, label="Number of vertices")

axes[0].set_ylabel("Mean track momentum [GeV]")

Signal sits towards the upper left: fewer tracks, but each carrying more momentum.
Background sits lower right: many tracks sharing the energy between them.

## 9. Which variables say the same thing?

If two columns are strongly correlated they carry nearly the same information, and
including both adds little. `.corr()` computes the correlation between every pair.

In [ ]:
features = [c for c in df.columns if c not in ("interaction", "weight")]
correlation = df[features].corr()

fig, ax = plt.subplots(figsize=(9, 8), layout="constrained")
image = ax.imshow(correlation, cmap="RdBu_r", vmin=-1, vmax=1)

ax.set_xticks(range(len(features)), features, rotation=90)
ax.set_yticks(range(len(features)), features)
fig.colorbar(image, ax=ax, label="Correlation")

Deep red means "these two go up together", deep blue means "when one goes up the other
goes down". Notice the block of `n_vtrk`, `n_vtrk_ip5`, `n_vtrk_100mrad` and `n_kinks` in
the top left - they are all counting roughly the same thing.

## 10. Missing values

Not every quantity could be measured for every vertex. Missing entries are `NaN`
("not a number"), and pandas skips them in `mean()`, `hist()` and so on.

In [ ]:
missing = df.isna().mean() * 100
missing[missing > 0].round(2).sort_values(ascending=False)

Where these come from:

- **`p_max`, `p_sum`, `p_mean`** - momentum can only be measured for a track long enough
  to be followed through many films. About a quarter of tracks are too short, and in a
  few vertices *no* track could be measured. `n_tracks_with_momentum` tells you how many
  went into the average.
- **`delta_phi_p_mean`, `delta_phi_p_max`** - these need momenta, so they need at least
  two measurable tracks.
- **`kink_angle_max`** - only defined if a kink (a sudden change of direction, meaning a
  particle decayed) was found at all.
- **`dphi_p`** - weights the other tracks by their momentum, so it is missing when none
  of them has one.

Missing data is never random. Always ask *why* something is missing before deciding what
to do about it.

## 11. Machine learning with scikit-learn

So far *you* have been looking at histograms and deciding which variables separate signal
from background. A **machine learning model** does the same thing automatically: it is
shown many examples where the answer is known, and learns a rule that turns the input
columns into a prediction.

This section does not build a finished classifier - that part is up to you. Instead it
walks through the steps every machine learning project needs, using
[scikit-learn](https://scikit-learn.org/stable/), the standard Python library for it.

scikit-learn expects the data in two pieces:

- **`X`**, the **features**: a table with one row per example and one column per input
  variable. These are the only things the model is allowed to see.
- **`y`**, the **target**: the correct answer for each row. Here that is `is_signal`,
  which we made in section 3.

Two columns must be left out of `X`. `interaction` *is* the answer, so including it
would be cheating. `weight` gives the answer away too, as section 4 showed.

In [ ]:
X = df.drop(columns=["interaction", "weight"])
y = is_signal
weights = df["weight"]

print(f"X has {X.shape[0]} rows and {X.shape[1]} feature columns")

## 12. Splitting into a training set and a test set

Imagine studying for an exam using the exact questions that will be on it. You would get
full marks, but that says nothing about whether you understand the subject. A model is
the same: if we test it on the rows it learned from, it can score perfectly just by
memorising them. This is called **overfitting**.

So before doing anything else we set some rows aside:

- the **training set** is what the model learns from;
- the **test set** is locked away and only used at the very end, to check how well the
  model does on data it has never seen.

`train_test_split` shuffles the rows and cuts them in two. A few of its options matter:

- `test_size=0.25` keeps a quarter of the rows for testing.
- `stratify=y` makes sure both parts have the same fraction of signal, so neither part
  is accidentally short of one class.
- `random_state=42` fixes the shuffle, so you get the same split every time you run the
  notebook. Any number works (can you guess why 42 is commonly used?).

We pass `weights` in as well, so each row keeps its weight on whichever side it lands.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights, test_size=0.25, stratify=y, random_state=42
)

print(f"training set: {len(X_train)} rows")
print(f"test set    : {len(X_test)} rows")

**From now on, do not look at the test set** until you have a final model. Every decision
- which columns to use, how to clean them, which model to pick - should be made using the
training set only. Otherwise information from the test set leaks into your choices, and
the final score will look better than the model really is.

## 13. Preparing the data

Most models cannot deal with the data exactly as it is. Two common problems:

1. **Missing values.** Many scikit-learn models stop with an error if they meet a `NaN`.
   Section 10 showed which columns have gaps.
2. **Very different scales.** `p_sum` can be in the hundreds of GeV, while `slope_mean`
   is usually below 0.1. Some models treat big numbers as more important, just because
   they are big. **Normalising** puts every column on a similar scale.

Here is how to fix the first problem for **one** column, `p_mean`, which is missing when
no track in the vertex had a measurable momentum. One simple choice is to fill the gaps
with the **median** of the column.

Notice the pattern, because it is the same for every preprocessing step in scikit-learn:

- `.fit(...)` *learns* something from the data - here, what the median is. It is only
  ever called on the **training** set.
- `.transform(...)` *applies* what was learned. It is called on both sets, so the test
  set is filled with the median of the *training* set.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
imputer.fit(X_train[["p_mean"]])

X_train["p_mean"] = imputer.transform(X_train[["p_mean"]]).ravel()
X_test["p_mean"] = imputer.transform(X_test[["p_mean"]]).ravel()

print(f"median used to fill the gaps: {imputer.statistics_[0]:.1f} GeV")
print(f"missing values left in p_mean: {X_train['p_mean'].isna().sum()}")

Filling with the median is not the only option, and not always the best one. You could
also drop the rows with gaps, fill with a value that can never occur (like `-1`), or
choose a model that handles `NaN` by itself. Each choice has consequences - think back
to the end of section 10: missing data is never random.

Normalising works the same way. `StandardScaler` shifts and stretches a column so that
it has mean 0 and standard deviation 1, and it also uses `.fit()` on the training set
and `.transform()` on both.

### Your turn

**Which other columns might need preprocessing and cleaning before passing them to a
machine learning model?**

Some things to think about:

- Which columns still contain `NaN`? Is the median a sensible fill value for each of
  them, or does the *reason* the value is missing suggest something better?
- Which columns have very large or very small numbers compared to the others? Look at
  `X_train.describe()`.
- Are there columns that are nearly copies of each other (section 9)? Does the model
  need all of them?
- Does your choice of model (next section) change the answer to any of these questions?

The scikit-learn guide to
[preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html) and to
[imputation of missing values](https://scikit-learn.org/stable/modules/impute.html)
describes the tools available.

## 14. Choosing a model

scikit-learn has many classification models. Here is the full list:
<https://scikit-learn.org/stable/supervised_learning.html>

In [ ]:
# Write your code here! :D

## 15. Evaluating a model

Once a model is trained, it makes a prediction for every row of the test set, and we
compare those predictions to the true answers in `y_test`. To show how this works
without building a model, we use a hand-made rule from section 6 instead: *call a vertex
signal if it has 8 or fewer tracks*. A trained model would give you `y_pred` in exactly
the same form.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

# A simple cut, just to show how to use scikit-learn
y_pred = X_test["n_vtrk"] <= 8

print(f"accuracy: {accuracy_score(y_test, y_pred):.3f}")
confusion_matrix(y_test, y_pred)

**Accuracy** is the fraction of rows predicted correctly. It is easy to understand but
can be misleading: if 90% of the rows were background, a "model" that always says
"background" would get 90% accuracy while finding no neutrinos at all.

The **confusion matrix** gives the full picture. Its rows are the true class and its
columns the predicted class, both in the order *background, signal*:

|  | predicted background | predicted signal |
|---|---|---|
| **true background** | correctly rejected | background that sneaks in |
| **true signal** | neutrinos we lost | neutrinos we found |

For a physicist the two numbers that matter most are the **signal efficiency** (what
fraction of the real neutrinos did we keep?) and the signal purity, i.e. the amount of **background that sneaks
in**.

And remember section 7: in the real experiment background is far more common than in
the simulation. Passing the weights with `sample_weight=` counts each row as the number
of real events it represents, which answers the question that actually matters: *how
many neutrinos, and how many fake ones, would we see in the real detector?*

In [ ]:
matrix = confusion_matrix(y_test, y_pred, sample_weight=w_test)
labels = ["background", "signal"]

fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")
image = ax.imshow(matrix, cmap="Blues")

# Write the number inside each square, in white on the dark squares so it stays readable
for true in range(2):
    for predicted in range(2):
        value = matrix[true, predicted]
        ax.text(predicted, true, f"{value:.1f}", ha="center", va="center",
                color="white" if value > matrix.max() / 2 else "black")

ax.set_xticks([0, 1], labels)
ax.set_yticks([0, 1], labels)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Weighted to the real detector")
fig.colorbar(image, ax=ax, label="Expected vertices")

Compare the two matrices. A rule that looks decent on the simulated numbers can let in
far more background than signal once the weights are applied.

Other useful tools in `sklearn.metrics` are `precision_score`, `recall_score` (another
name for signal efficiency) and `roc_curve`, which shows how efficiency and background
change as you move the cut. The scikit-learn page on
[model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)
explains them all.

## Where to go next

You have seen that signal and background differ in several variables. The next step is to
let a computer combine them into a single decision - that is
**machine learning**. Sections 11-15 laid out the steps; choosing, training and
tuning a model is up to you.

Before that, try these:

1. Plot every feature as a signal/background overlay. Which three separate best?
2. Does `nueCC` look different from `numuCC`? (Filter with `df[df["interaction"] == "nueCC"]`.)

### Documentation

Everything used here is documented online. These are the official references:

- **pandas** - tables, `read_csv`, filtering, `groupby`: <https://pandas.pydata.org/docs/>
  (start with the [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) guide)
- **NumPy** - arrays and numerical maths: <https://numpy.org/doc/stable/>
  (start with [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html))
- **Matplotlib** - plotting: <https://matplotlib.org/stable/>
  (the [pyplot tutorial](https://matplotlib.org/stable/tutorials/pyplot.html) and the
  [example gallery](https://matplotlib.org/stable/gallery/index.html) are the most useful pages)
- **scikit-learn** - machine learning: <https://scikit-learn.org/stable/>
  (start with [An introduction to machine learning](https://scikit-learn.org/stable/tutorial/basic/tutorial.html))